### Determine current BTC market

In [ ]:
from time import time
from datetime import datetime
from zoneinfo import ZoneInfo
from datetime import timezone

WINDOW_SECS = 300  # 5-min window

def current_window_start() -> int:
    """
    Unix timestamp of the current 5-min window's start.
    """
    return (int(time()) // WINDOW_SECS) * WINDOW_SECS


def to_EST(ts: int) -> str:
    dt = datetime.fromtimestamp(ts, tz=timezone.utc).astimezone(ZoneInfo("America/New_York"))
    return dt.strftime("%Y-%m-%d %I:%M:%S %p EST")


def current_window_slug(ts) -> str:
    """
    Current 5-min window slug.
    """
    return f"btc-updown-5m-{ts}"


start = current_window_start()
start_EST = to_EST(start)
slug = current_window_slug(start)
print([start, start_EST, slug])

### Init "clob" Client

In [ ]:

from py_clob_client.client import ClobClient
from config import load_config

config = load_config()

host = "https://clob.polymarket.com"
chain_id = 137  # Polygon mainnet

# Derive API credentials (L1 → L2 auth)
temp_client = ClobClient(host, key=config.polymarket.private_key, chain_id=chain_id)
api_creds = temp_client.create_or_derive_api_creds()

# Initialize trading client
client = ClobClient(
    host,
    key=config.polymarket.private_key,
    chain_id=chain_id,
    creds=api_creds,
    # I'm going with the proxy wallet through polymarket to avoid paying gas fees. This seemed like the best
    # one to use from: https://docs.polymarket.com/trading/overview#signature-types.
    signature_type=1,
    funder=config.polymarket.wallet_address,
)
client

### Get active BTC up / down Market

In [ ]:
from requests import get as GET
from json import loads
from requests.exceptions import HTTPError

class MarketNotFound(Exception):
    ...

def get_market_by_slug(slug: str) -> dict:
    """Fetch a single BTC 5-min market by its exact slug."""
    response = GET(f"https://gamma-api.polymarket.com/events", params={"slug": slug}, timeout=10)
    try:
        response.raise_for_status()
        data = response.json()
        if data is None or not isinstance(data, list) or len(data) < 1:
            raise MarketNotFound(f"No market data found for slug: {slug}.")
        return data[0]
    except HTTPError as e:
        raise MarketNotFound from e
    

market = get_market_by_slug(slug)
if len(market["markets"]) != 1:
    raise AssertionError("Expected BTC Up/Down market response to contain exactly 1 market!")
outcomes = loads(market["markets"][0]["outcomes"])
tids = loads(market["markets"][0]["clobTokenIds"])
market_clobs = dict(zip([x.lower() for x in outcomes], tids))
market_clobs

In [ ]:
up_book = client.get_order_book(market_clobs["down"])
up_book.asks

In [ ]:
from py_clob_client.clob_types import OrderType
from time import sleep

def in_buy_threshold(price: float, min=0.9, max=0.95) -> bool:
    return price >= min and price <= max

while True:
    up_price = client.calculate_market_price(
        token_id=market_clobs["up"],
        side="BUY",
        amount=10,
        order_type=OrderType.FOK,  # type: ignore
    )
    down_price = client.calculate_market_price(
        token_id=market_clobs["down"],
        side="BUY",
        amount=10,
        order_type=OrderType.FOK,  # type: ignore
    )
    if in_buy_threshold(up_price):
        ...
    elif in_buy_threshold(down_price):
        ...
    print({"Up": estimated_up_price, "Down": estimated_down_price})
    sleep(0.1)